In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/sanilawijesekara/sl-paddy-combined/sl_district_paddy.csv


In [4]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
import joblib

# 1. Load Data
df = pd.read_csv("/kaggle/input/datasets/sanilawijesekara/sl-paddy-combined/sl_district_paddy.csv")
df.drop(columns=['unnamed: 0','Unnamed: 0.1','Unnamed: 0', 'average_yield'], inplace=True, errors='ignore')

# 2. Encoding & Cleaning
le_district = LabelEncoder()
df['district_id'] = le_district.fit_transform(df['district'])
hazard_cols = ['hazard_drought', 'hazard_flood', 'hazard_heavy_rain', 'hazard_landslide', 'hazard_lightning', 'hazard_wind']
df[hazard_cols] = df[hazard_cols].astype(int)
if 'hazard_heat' in df.columns: df.drop(columns=['hazard_heat'], inplace=True)

# 3. Features Definition
ts_features = ['ndvi_median_smooth', 'lswi_median_smooth', 'evi_median_smooth', 'ndwi_median_smooth', 'bsi_median_smooth', 'ndvi_vel_z', 'lswi_vel_z', 'rain_7d_mean', 'rain_14d_mean', 'tmean_mean','bsi_z', 'rh_mean_mean', 'delta_days', 'doy_sin', 'doy_cos']
static_features = ['lat', 'lon', 'elevation', 'slope', 'district_id']
target_cols = ['stage', 'ndvi_zscore', 'cpi'] + hazard_cols
all_required_cols = ['pixel_id', 'date', 'year', 'month', 'season', 'district'] + ts_features + static_features + target_cols

df_lstm = df[all_required_cols].copy()
df_lstm['date'] = pd.to_datetime(df_lstm['date'])

# 4. Vectorized Season & Cycle Logic
year_str = df_lstm['year'].astype(str)
next_year_str = (df_lstm['year'] + 1).astype(str)
prev_year_str = (df_lstm['year'] - 1).astype(str)
months = df_lstm['month'].values

cond_yala = (months >= 5) & (months <= 8)
cond_maha_part2 = (months <= 3)
cond_maha_part1 = ~cond_yala & ~cond_maha_part2

df_lstm['season'] = np.where(cond_yala, 'Yala', 'Maha')
conditions = [cond_yala, cond_maha_part2, cond_maha_part1]
choices = [year_str + '_Yala', prev_year_str + '_' + year_str + '_Maha', year_str + '_' + next_year_str + '_Maha']
df_lstm['cycle_id'] = np.select(conditions, choices, default=year_str + '_Maha')

# 5. Scaling
raw_temporal = ['ndvi_median_smooth', 'lswi_median_smooth', 'evi_median_smooth', 'ndwi_median_smooth', 'bsi_median_smooth', 'rain_7d_mean', 'rain_14d_mean', 'tmean_mean', 'rh_mean_mean', 'delta_days']
raw_static = ['elevation', 'slope','lat','lon']
features_to_scale = raw_temporal + raw_static

scaler = StandardScaler()
df_lstm[features_to_scale] = scaler.fit_transform(df_lstm[features_to_scale])
df_lstm['is_growing'] = df_lstm['stage'].isin([1, 2, 3]).astype('float32')

# 6. SAVE PREPROCESSED DATA
print("💾 Saving preprocessed data to /kaggle/working/...")
df_lstm.to_parquet('/kaggle/working/preprocessed_pixel_data.parquet')
joblib.dump(scaler, '/kaggle/working/main_scaler.pkl')
joblib.dump(le_district, '/kaggle/working/district_encoder.pkl')
print("✅ Done! Files saved: preprocessed_pixel_data.parquet, main_scaler.pkl")

💾 Saving preprocessed data to /kaggle/working/...
✅ Done! Files saved: preprocessed_pixel_data.parquet, main_scaler.pkl
